In [10]:
from lxml import html
import requests
import numpy as np
import json, requests
import datetime
from datetime import timedelta
import time
import pandas as pd


In [11]:
sysdate = datetime.datetime.fromtimestamp(int(time.time())).strftime('%Y-%m-%d %H:%M:%S')
print(sysdate)

file_historical = open('../../_data/datasets/bitcoin/historical_price.csv', 'w')

#all headers
header_historical = ['id', 'date', 'open', 'high', 'low', 'close', 'volume', 'market_cap']

# Write headers into files
file_historical.write(','.join(str(e) for e in header_historical) + '\n')

2020-02-09 17:48:55


46

In [12]:
print((datetime.datetime.fromtimestamp(int(time.time())) - timedelta(days=1)).strftime('%Y%m%d'))

20200208


In [13]:
id = 'bitcoin'
# Historical Data
historical_data_url = 'https://coinmarketcap.com/currencies/' + id + '/historical-data/?start=20100101&end=' + (datetime.datetime.fromtimestamp(int(time.time())) - timedelta(days=2)).strftime('%Y%m%d')
#historical_data_url = 'https://coinmarketcap.com/currencies/bitcoin/historical-data/?start=20200101&end=20200125'

page = requests.get(historical_data_url)
tree = html.fromstring(page.content)
historical_data = tree.xpath('//td/div/text()')
try:
    historical_data_table = np.reshape(np.array(historical_data), (-1, 7))
    for data_historical in historical_data_table:
        date = datetime.datetime.strptime(data_historical[0], "%b %d, %Y").date()
        row_history = [id, date.strftime('%Y-%m-%d'),
                       float(data_historical[1].replace('-', '').replace(',', '') or 0.0),
                       float(data_historical[2].replace('-', '').replace(',', '') or 0.0),
                       float(data_historical[3].replace('-', '').replace(',', '') or 0.0),
                       float(data_historical[4].replace('-', '').replace(',', '') or 0.0),
                       int(data_historical[5].replace('-', '').replace(',', '') or 0),
                       int(data_historical[6].replace('-', '').replace(',', '') or 0)]
        file_historical.write(','.join(str(e) for e in row_history) + '\n')
except Exception as e:
    print("Unexpected error", id, e )
    
file_historical.close()

In [14]:
df = pd.read_csv("../../_data/datasets/bitcoin/historical_price.csv")

Adding Day of the week to the dataframe

In [15]:
df['weekday'] =pd.to_datetime(df['date']).apply(lambda x: x.weekday())


In [16]:
df_invest_period = df[df['date'] >= '2018-09-01']
df_invest_period = df_invest_period.drop(['id','open','high','low','volume','market_cap'], 1)
df_invest_period['investment'] = df_invest_period.apply(lambda row: 100 if row.weekday == 0 else 0, axis=1)
df_invest_period['fee'] = df_invest_period.apply(lambda row: 4 if row.weekday == 0 else 0, axis=1)
df_invest_period['cc_fee'] = df_invest_period.apply(lambda row: 7 if row.weekday == 0 else 0, axis=1)
df_invest_period['gross_investment'] = df_invest_period.apply(lambda row: row.investment + row.cc_fee if row.weekday == 0 else 0, axis=1)
df_invest_period['net_investment'] = df_invest_period.apply(lambda row: row.investment - row.fee if row.weekday == 0 else 0, axis=1)
df_invest_period['bitcoin_balance'] = df_invest_period.apply(lambda row: row.net_investment/row.close if row.weekday == 0 else 0, axis=1)
df_invest_period

,date,close,weekday,investment,fee,cc_fee,gross_investment,net_investment,bitcoin_balance
0,2020-02-08,9865.12,5,0,0,0,0,0,0.000000
1,2020-02-07,9795.94,4,0,0,0,0,0,0.000000
2,2020-02-06,9729.80,3,0,0,0,0,0,0.000000
3,2020-02-05,9613.42,2,0,0,0,0,0,0.000000
4,2020-02-04,9180.96,1,0,0,0,0,0,0.000000
5,2020-02-03,9293.52,0,100,4,7,107,96,0.010330
6,2020-02-02,9344.37,6,0,0,0,0,0,0.000000
7,2020-02-01,9392.88,5,0,0,0,0,0,0.000000
8,2020-01-31,9350.53,4,0,0,0,0,0,0.000000
9,2020-01-30,9508.99,3,0,0,0,0,0,0.000000


In [17]:
#print(df_invest_period.date.max())
#last_date = df_invest_period.date.max()
#print(df_invest_period.loc[df_invest_period['date'] == last_date, 'close'])
last_price = df_invest_period.iloc[0]['close']

#print()

In [18]:
print(df_invest_period['gross_investment'].sum())
print(df_invest_period['gross_investment'].sum()/107)
print(df_invest_period['bitcoin_balance'].sum()*last_price)
print('ROI: ', df_invest_period['bitcoin_balance'].sum()*last_price/df_invest_period['gross_investment'].sum())



8025
75.0
11587.24697375027
ROI:  1.4438937038941146
